# 03 - Model Training

Train LightGBM with holdout validation, feature pruning, and K-fold ensembling. Save model and evaluation plots.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from src import config
from src.evaluation import plot_feature_importance, plot_roc_curve
from src.model_training import evaluate_models, train_kfold_lightgbm_ensemble
from src.utils import ensure_dir, save_pickle

In [ ]:
merged_train = Path(config.MERGED_TRAIN_PATH)
if not merged_train.exists():
    raise FileNotFoundError('Run notebooks/02_feature_engineering.ipynb first.')

df_train = pd.read_pickle(merged_train)
print('Loaded merged train:', df_train.shape)

In [ ]:
validation_ev = evaluate_models(
    df_train,
    random_state=config.RANDOM_STATE,
    ohe_max_categories=config.OHE_MAX_CATEGORIES,
    miss_drop_threshold=config.MISS_DROP_THRESHOLD,
    prune_bottom_frac=config.PRUNE_BOTTOM_FRAC,
)
print('Holdout AUC (LGBM):', round(validation_ev['auc_lgb'], 5))
print('Holdout AUC (LogReg):', round(validation_ev['auc_lr'], 5))
print('Best iteration:', validation_ev['best_iteration'])

In [ ]:
bundle = train_kfold_lightgbm_ensemble(
    df_train,
    feature_keep_indices=validation_ev['feature_keep_indices'],
    n_splits=config.N_FOLD_ENSEMBLE,
    random_state=config.RANDOM_STATE,
)
print('OOF AUC:', round(bundle['oof_auc'], 5))
print('Fold AUCs:', [round(x, 5) for x in bundle['fold_val_aucs']])

In [ ]:
ensure_dir(config.MODELS_DIR)
ensure_dir(config.FIGURES_DIR)

save_pickle(bundle, config.MODEL_BUNDLE_PATH)
plot_roc_curve(validation_ev['y_val'], validation_ev['val_pred'])
plot_feature_importance(bundle['models'][-1], bundle['feature_names_final'])

print(f'Saved model bundle: {config.MODEL_BUNDLE_PATH}')
print(f'Saved ROC curve: {config.ROC_CURVE_PATH}')
print(f'Saved feature importance: {config.FEATURE_IMPORTANCE_PATH}')